# Load Library

In [1]:
import json
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
transformers.logging.set_verbosity_error()

# Load data

In [2]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [3]:
test_data = load_json("/kaggle/input/task2-private/data/alqac25_private_test_task2.json")    
law_data = load_json("/kaggle/input/task2-private/data/alqac25_law.json")

# Get content from Article & Law ID

In [4]:
law_map = {}

for law in law_data:
    for article in law.get("articles", []):
        law_map[(law["id"], article["id"])] = article["text"]

In [5]:
def get_law_content(law_infors):
    law_content = ""
    for law_info in law_infors:
        law_id = law_info['law_id']
        article_id = law_info['article_id']
        law_content += f"Luật: {law_id}" + '\n' + law_map.get((law_id, article_id), "") + '\n'*2
    return law_content

# Design prompts for each type of question

In [6]:
def prompt_design_v2(question):
    # Extract relevant law articles from the question input
    law_infors = question['relevant_articles']

    # Retrieve the full content of the referenced legal articles
    law_content = get_law_content(law_infors)

    # Format prompt depending on the type of question
    if question['question_type'] == 'Đúng/Sai':
        return (
            f"Bạn là một chuyên gia trả lời câu hỏi nhận định đúng/sai pháp luật."
            f"Dựa vào điều luật được cung cấp, hãy xác định xem nhận định trong câu hỏi dưới đây là Đúng hay Sai.\n\n"
            f"**YÊU CẦU BẮT BUỘC:**\n"
            f"Câu trả lời của bạn CHỈ ĐƯỢC PHÉP là MỘT trong hai từ sau: \"Đúng\" hoặc \"Sai\".\n\n"
            f"--- BẮT ĐẦU DỮ LIỆU ---\n\n"
            f"**Điều luật:**\n{law_content}\n\n"
            f"**Câu hỏi:**\n{question['text']}\n\n"
            f"--- KẾT THÚC DỮ LIỆU ---\n\n"
            f"**Nhận định trên là (chỉ điền Đúng hoặc Sai):**" 
        )

    elif question['question_type'] == 'Trắc nghiệm':
        choices = question['choices']
        return (
            "Bạn là một chuyên gia trả lời câu hỏi trắc nghiệm pháp luật. Nhiệm vụ của bạn là đọc kỹ điều luật và câu hỏi, sau đó chọn một đáp án duy nhất (A, B, C, hoặc D).\n\n"
            "**YÊU CẦU BẮT BUỘC:** Câu trả lời cuối cùng của bạn phải là MỘT KÝ TỰ DUY NHẤT.\n\n"
            "--- Bối cảnh ---\n"
            f"Điều luật: {law_content}\n\n"
            "--- Câu hỏi và Lựa chọn ---\n"
            f"Câu hỏi: {question['text']}\n"
            f"A. {choices['A']}\n"
            f"B. {choices['B']}\n"
            f"C. {choices['C']}\n"
            f"D. {choices['D']}\n\n"
            f"--- Đáp án ---\n"
            f"Lựa chọn chính xác nhất là (chỉ ghi A, B, C, hoặc D):" 
        )

    else: # Free-text question
        return (
            "Bạn là một trợ lý pháp lý chuyên trích xuất thông tin. Nhiệm vụ của bạn là đọc điều luật được cung cấp và trả lời câu hỏi một cách ngắn gọn, chính xác.\n\n"
            "### YÊU CẦU NGHIÊM NGẶT:\n"
            "1. Chỉ trích xuất phần nội dung trả lời trực tiếp cho câu hỏi trong điều luật.\n"
            "2. KHÔNG được sao chép toàn bộ đoạn luật, chỉ lấy phần có ý nghĩa trả lời.\n"
            "3. KHÔNG được viết lại câu hỏi, không được thêm giải thích hoặc bình luận.\n"
            "4. Câu trả lời phải ngắn gọn, đầy đủ ý và đúng theo điều luật.\n\n"
            "<DỮ LIỆU>\n"
            f"Điều luật:\n{law_content}\n\n"
            f"Câu hỏi:\n{question['text']}\n\n"
            "<Câu trả lời (chỉ một câu ngắn, không dài dòng, không lặp lại thông tin)> \n"
        )


# Get answer from LLMs

In [7]:
def predict_answer(model, tokenizer, question):
    # Generate the input prompt based on the question content and type
    input_text = prompt_design_v2(question)
    
    # Tokenize input
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Configure generation settings based on the type of question
    qtype = question['question_type']

    if qtype == "Đúng/Sai":
        do_sample = False
        max_new_tokens = 3  # Enough to generate "Đúng" or "Sai"

    elif qtype == "Trắc nghiệm":
        do_sample = False
        max_new_tokens = 3  # Enough to generate "A", "B", "C", "D"

    else:
        do_sample = True
        max_new_tokens = 128  # Sufficient length for concise but complete answers

        
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the generated part (exclude input)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
    return response


# Model & Tokenizer Loading

In [8]:
# 1: AITeamVN/GRPO-VI-Qwen2-7B-RAG
# 2: Qwen/Qwen2.5-7B-Instruct
# 3: AITeamVN/Vi-Qwen2-7B-RAG

## Please replace model_name one by one with the names of the three models above, then run all to generate the three result files.

In [9]:
model_name = "AITeamVN/GRPO-VI-Qwen2-7B-RAG"

In [10]:
def load_model(model_name):
    print(f"Loading {model_name} model...")
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        use_cache=True
    )
    
    # Set pad token if not exists
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print(f"Model {model_name} loaded successfully!")
    print(f"Model device: {model.device}")
    print(f"Model dtype: {model.dtype}")

    return model, tokenizer

In [11]:
model, tokenizer = load_model(model_name)

Loading AITeamVN/GRPO-VI-Qwen2-7B-RAG model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model AITeamVN/GRPO-VI-Qwen2-7B-RAG loaded successfully!
Model device: cuda:0
Model dtype: torch.float16


# Pipeline

In [12]:
def run_pipeline(model, tokenizer):
    results = []
    # Iterate over all questions in the test dataset with a progress bar
    for question in tqdm(test_data, desc="Processing and Evaluating questions"):
        try:
            # 1. Generate prediction from the model
            pred_answer = predict_answer(model, tokenizer, question)
            
            # 2. Store the result with relevant metadata
            results.append({
                "question_id": question["question_id"],
                "question": question["text"],
                "question_type": question["question_type"],
                "predicted_answer": pred_answer,
            })
            
        except Exception as e:
            print(f"Lỗi trong pipeline với câu hỏi {question.get('question_id', 'unknown')}: {str(e)}")
            results.append({
                "question_id": question.get("question_id", "unknown"),
                "question": question["text"],
                "question_type": question["question_type"],
                "predicted_answer": "ERROR",
            })
    
    return results

# RUN

In [13]:
# Run the pipeline
print("Starting evaluation pipeline...")
final_results = run_pipeline(model, tokenizer)

# Save results
output_file = f"{model_name.split('/')[-1]}_private.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

print(f"Evaluation completed! Results saved to {output_file}")
print(f"Total questions processed: {len(final_results)}")

Starting evaluation pipeline...


Processing and Evaluating questions: 100%|██████████| 82/82 [13:34<00:00,  9.94s/it]

Evaluation completed! Results saved to GRPO-VI-Qwen2-7B-RAG_private.json
Total questions processed: 82


In [14]:
# Print some sample results
print("\nSample results:")
for i, result in enumerate(final_results[:3]):
    print(f"\nQuestion {i+1}:")
    print(f"Type: {result['question_type']}")
    print(f"Question: {result['question'][:100]}...")
    print(f"Predicted: {result['predicted_answer']}")


Sample results:

Question 1:
Type: Đúng/Sai
Question: Hợp đồng điện tử được ký kết giữa hai hệ thống thông tin tự động mà không có sự can thiệp của con ng...
Predicted: Sai

---

Question 2:
Type: Trắc nghiệm
Question: Khi nào áp dụng Luật Giao dịch điện tử 2023?...
Predicted: D

---

Question 3:
Type: Trắc nghiệm
Question: Các loại hình giao dịch điện tử của cơ quan nhà nước...
Predicted: D

---
